In [1]:
library(lme4)
library(rstanarm)
library(emmeans)
library(dplyr)

Loading required package: Matrix

Loading required package: Rcpp

This is rstanarm version 2.32.2

- See https://mc-stan.org/rstanarm/articles/priors for changes to default priors!

- Default priors may change, so it's safest to specify priors, even if equivalent to the defaults.

- For execution on a local, multicore CPU with excess RAM we recommend calling

  options(mc.cores = parallel::detectCores())

Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
df <- read.csv("processed_data/acceptance_data_for_logistic_and_probit.csv") %>%
  mutate(
    # Convert categorical variables to factors
    tone = as.factor(tone),
    lang = as.factor(lang),
    word = as.factor(word),
    participant = as.factor(participant),
    word_log_freq_z = as.numeric(log(word_freq)),
    
    acceptance = as.integer(as.logical(acceptance)) 
  )

In [3]:
# z-score word_log_freq within each language
df <- df %>%
  group_by(lang) %>%
  mutate(word_log_freq_z = scale(word_log_freq_z)) %>%
  ungroup()

In [4]:
library(dplyr)
library(emmeans)
library(rstanarm)

# 1. Ensure the output directory exists BEFORE the loop starts
output_dir <- "processed_data/thresholds_probit_per_participant"
if (!dir.exists(output_dir)) {
  dir.create(output_dir, recursive = TRUE)
}

# Initialize an empty list to also store the merged dataframe
results_list <- list()

# 2. Loop through the languages
for (current_lang in c("min", "man")) {
  
  participants <- unique(df$participant[df$lang == current_lang])
  
  for (current_participant in participants) {
    
    sub_df <- df %>% 
      filter(lang == current_lang, participant == current_participant)
    
    if (nrow(sub_df) < 10) next
    
    tryCatch({
      
      # Added init = "0" to help prevent the initialization crash
      model <- stan_glmer(acceptance ~ level * tone * word_log_freq_z + (1 | word), 
                        data = sub_df, 
                        family = binomial(link = "probit"),
                        init = "0",
                        adapt_delta = 0.99,
                        refresh = 0)
      
      emm_intercepts <- as.data.frame(emmeans(model, ~ tone, at = list(level = 0)))
      emm_slopes <- as.data.frame(emtrends(model, ~ tone, var = "level"))
      
      thresholds_df <- emm_intercepts %>%
        select(tone, intercept = emmean) %>%
        left_join(emm_slopes %>% select(tone, slope = level.trend), by = "tone") %>%
        mutate(threshold_50 = - (intercept / slope))
      
      thresholds_df$lang <- current_lang
      thresholds_df$participant <- current_participant
      
      # ---------------------------------------------------------
      # NEW: Save this specific participant's data immediately
      # ---------------------------------------------------------
      # Creates a filename like: thresholds_probit_per_participant/man_P3.csv
      file_name <- sprintf("%s/%s_%s.csv", output_dir, current_lang, current_participant)
      write.csv(thresholds_df, file_name, row.names = FALSE)
      
      # Still append it to the list so you get the giant merged CSV at the end
      results_list[[length(results_list) + 1]] <- thresholds_df
      
      message("Success: ", current_participant, " in ", current_lang)
      
    }, error = function(e) {
      message("Failed: ", current_participant, " in ", current_lang, " - ", e$message)
    })
  }
}

# 6. Bind the list of dataframes together into one dataframe
final_thresholds_df <- bind_rows(results_list)

# 7. Reorder columns
final_thresholds_df <- final_thresholds_df %>% 
  relocate(participant, lang, tone, threshold_50, slope, intercept)

# 8. Save the massive master CSV
write.csv(final_thresholds_df, "processed_data/all_participant_thresholds_probit.csv", row.names = FALSE)

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions

Success: P1 in min

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions

Success: P2 in min

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions

Success: P7 in min

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions

Success: P14 in min

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions

Success: P15 in min

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions

Success: P16 in min

NOTE: Results may be misleading due to involvement in interac

In [5]:
library(dplyr)
library(emmeans)
library(rstanarm)

# 1. Ensure the output directory exists BEFORE the loop starts
output_dir <- "processed_data/thresholds_probit_per_participant"
if (!dir.exists(output_dir)) {
  dir.create(output_dir, recursive = TRUE)
}

# Initialize list and set up the text file for significance tracking
results_list <- list()
txt_filename <- "processed_data/probit_frequency_significance.txt"

# Create the text file and write the header row
cat("participant\tlang\tfreq_crosses_zero\tinteraction_crosses_zero\n", file = txt_filename)

# 2. Loop through the languages
for (current_lang in c("min", "man")) {
  
  participants <- unique(df$participant[df$lang == current_lang])
  
  for (current_participant in participants) {
    
    sub_df <- df %>% 
      filter(lang == current_lang, participant == current_participant)
    
    if (nrow(sub_df) < 10) next
    
    tryCatch({
      
      # Added init = "0" to help prevent the initialization crash
      model <- stan_glmer(acceptance ~ level * tone * word_log_freq_z + (1 | word), 
                        data = sub_df, 
                        family = binomial(link = "probit"),
                        init = "0",
                        adapt_delta = 0.99,
                        refresh = 0)
      
      # ---------------------------------------------------------
      # NEW: Check if the 95% Credible Intervals cross zero
      # ---------------------------------------------------------
      # Get the 95% boundaries for all parameters
      ci_95 <- posterior_interval(model, prob = 0.95)
      
      # Helper function to check if the interval straddles zero
      check_zero <- function(param) {
        if (param %in% rownames(ci_95)) {
          return(ci_95[param, "2.5%"] < 0 & ci_95[param, "97.5%"] > 0)
        } else {
          return(NA) # In case the parameter was dropped perfectly collinear
        }
      }
      
      freq_zero <- check_zero("word_log_freq_z")
      interaction_zero <- check_zero("level:word_log_freq_z")
      
      # Append this participant's results directly to the text file
      cat(sprintf("%s\t%s\t%s\t%s\n", 
                  current_participant, current_lang, freq_zero, interaction_zero), 
          file = txt_filename, append = TRUE)
      # ---------------------------------------------------------

      emm_intercepts <- as.data.frame(emmeans(model, ~ tone, at = list(level = 0)))
      emm_slopes <- as.data.frame(emtrends(model, ~ tone, var = "level"))
      
      thresholds_df <- emm_intercepts %>%
        select(tone, intercept = emmean) %>%
        left_join(emm_slopes %>% select(tone, slope = level.trend), by = "tone") %>%
        mutate(threshold_50 = - (intercept / slope))
      
      thresholds_df$lang <- current_lang
      thresholds_df$participant <- current_participant
      
      # Save this specific participant's CSV immediately
      file_name <- sprintf("%s/%s_%s.csv", output_dir, current_lang, current_participant)
      write.csv(thresholds_df, file_name, row.names = FALSE)
      
      # Append it to the list for the giant merged CSV
      results_list[[length(results_list) + 1]] <- thresholds_df
      
      message("Success: ", current_participant, " in ", current_lang)
      
    }, error = function(e) {
      message("Failed: ", current_participant, " in ", current_lang, " - ", e$message)
    })
  }
}

# 6. Bind the list of dataframes together
final_thresholds_df <- bind_rows(results_list)

# 7. Reorder columns
final_thresholds_df <- final_thresholds_df %>% 
  relocate(participant, lang, tone, threshold_50, slope, intercept)

# 8. Save the massive master CSV
write.csv(final_thresholds_df, "processed_data/all_participant_thresholds_probit.csv", row.names = FALSE)

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions

Success: P1 in min

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions

Success: P2 in min

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions

Success: P7 in min

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions

Success: P14 in min

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions

Success: P15 in min

NOTE: Results may be misleading due to involvement in interactions

NOTE: Results may be misleading due to involvement in interactions

Success: P16 in min

NOTE: Results may be misleading due to involvement in interac

In [7]:
sub_df <- df %>% 
      filter(lang == "min", participant == "P39")
model <- stan_glmer(acceptance ~ level * tone * word_log_freq_z + (1 | word), 
                          data = sub_df, 
                          family = binomial(link = "logit"),
                          adapt_delta = 0.99,
                          refresh = 0)
summary(model)


Model Info:
 function:     stan_glmer
 family:       binomial [logit]
 formula:      acceptance ~ level * tone * word_log_freq_z + (1 | word)
 algorithm:    sampling
 sample:       4000 (posterior sample size)
 priors:       see help('prior_summary')
 observations: 200
 groups:       word (9)

Estimates:
                                      mean   sd    10%   50%   90%
(Intercept)                           3.0    1.3   1.3   3.0   4.6
level                                -0.3    0.2  -0.5  -0.3  -0.1
tonetone51                          -10.0    1.9 -12.5 -10.0  -7.7
word_log_freq_z                       0.6    1.8  -1.6   0.6   2.9
level:tonetone51                      1.5    0.3   1.2   1.5   1.9
level:word_log_freq_z                 0.3    0.3  -0.1   0.3   0.6
tonetone51:word_log_freq_z           -1.5    1.9  -3.9  -1.5   0.9
level:tonetone51:word_log_freq_z     -0.3    0.3  -0.6  -0.3   0.1
b[(Intercept) word:早暗]              0.7    0.9  -0.2   0.5   1.9
b[(Intercept) word:水果]   

In [8]:
sub_df <- df %>% 
      filter(lang == "min", participant == "P41")
model <- stan_glmer(acceptance ~ level * tone * word_log_freq_z + (1 | word), 
                          data = sub_df, 
                          family = binomial(link = "logit"),
                          adapt_delta = 0.99,
                          refresh = 0)
summary(model)


Model Info:
 function:     stan_glmer
 family:       binomial [logit]
 formula:      acceptance ~ level * tone * word_log_freq_z + (1 | word)
 algorithm:    sampling
 sample:       4000 (posterior sample size)
 priors:       see help('prior_summary')
 observations: 200
 groups:       word (9)

Estimates:
                                      mean   sd    10%   50%   90%
(Intercept)                           2.0    1.0   0.7   2.0   3.2
level                                -0.2    0.1  -0.3  -0.2   0.0
tonetone51                          -10.0    1.8 -12.5  -9.9  -7.7
word_log_freq_z                      -0.3    1.6  -2.3  -0.3   1.7
level:tonetone51                      1.4    0.3   1.1   1.4   1.8
level:word_log_freq_z                 0.2    0.2  -0.1   0.2   0.5
tonetone51:word_log_freq_z           -0.1    1.7  -2.1   0.0   2.0
level:tonetone51:word_log_freq_z     -0.1    0.3  -0.5  -0.1   0.2
b[(Intercept) word:早暗]              0.5    0.5   0.0   0.3   1.2
b[(Intercept) word:水果]   

In [9]:
sub_df <- df %>% 
      filter(lang == "man", participant == "P12")
model <- stan_glmer(acceptance ~ level * tone * word_log_freq_z + (1 | word), 
                          data = sub_df, 
                          family = binomial(link = "logit"),
                          adapt_delta = 0.99,
                          refresh = 0)
summary(model)


Model Info:
 function:     stan_glmer
 family:       binomial [logit]
 formula:      acceptance ~ level * tone * word_log_freq_z + (1 | word)
 algorithm:    sampling
 sample:       4000 (posterior sample size)
 priors:       see help('prior_summary')
 observations: 200
 groups:       word (10)

Estimates:
                                      mean   sd    10%   50%   90%
(Intercept)                           7.8    1.5   6.0   7.7   9.7
level                                -1.2    0.2  -1.4  -1.1  -0.9
tonetone51                          -11.5    1.8 -13.9 -11.5  -9.2
word_log_freq_z                       0.6    1.0  -0.7   0.6   1.9
level:tonetone51                      2.0    0.3   1.6   2.0   2.4
level:word_log_freq_z                -0.1    0.1  -0.2  -0.1   0.1
tonetone51:word_log_freq_z           -2.3    2.7  -5.8  -2.2   1.2
level:tonetone51:word_log_freq_z      0.5    0.5  -0.1   0.5   1.1
b[(Intercept) word:今晚]              0.0    0.5  -0.5   0.0   0.5
b[(Intercept) word:功課]  

In [10]:
sub_df <- df %>% 
      filter(lang == "man", participant == "P13")
model <- stan_glmer(acceptance ~ level * tone * word_log_freq_z + (1 | word), 
                          data = sub_df, 
                          family = binomial(link = "logit"),
                          adapt_delta = 0.99,
                          refresh = 0)
summary(model)


Model Info:
 function:     stan_glmer
 family:       binomial [logit]
 formula:      acceptance ~ level * tone * word_log_freq_z + (1 | word)
 algorithm:    sampling
 sample:       4000 (posterior sample size)
 priors:       see help('prior_summary')
 observations: 200
 groups:       word (10)

Estimates:
                                      mean   sd    10%   50%   90%
(Intercept)                           7.6    1.4   5.9   7.5   9.4
level                                -1.2    0.2  -1.5  -1.2  -0.9
tonetone51                          -13.5    1.9 -16.0 -13.5 -11.2
word_log_freq_z                      -2.1    0.9  -3.3  -2.1  -1.0
level:tonetone51                      2.1    0.3   1.8   2.1   2.5
level:word_log_freq_z                 0.3    0.1   0.1   0.3   0.5
tonetone51:word_log_freq_z           -5.1    3.5  -9.7  -4.9  -0.6
level:tonetone51:word_log_freq_z      1.1    0.6   0.4   1.1   1.9
b[(Intercept) word:今晚]              0.2    0.5  -0.2   0.1   0.9
b[(Intercept) word:功課]  

In [11]:
sub_df <- df %>% 
      filter(lang == "man", participant == "P21")
model <- stan_glmer(acceptance ~ level * tone * word_log_freq_z + (1 | word), 
                          data = sub_df, 
                          family = binomial(link = "logit"),
                          adapt_delta = 0.99,
                          refresh = 0)
summary(model)


Model Info:
 function:     stan_glmer
 family:       binomial [logit]
 formula:      acceptance ~ level * tone * word_log_freq_z + (1 | word)
 algorithm:    sampling
 sample:       4000 (posterior sample size)
 priors:       see help('prior_summary')
 observations: 200
 groups:       word (10)

Estimates:
                                      mean   sd    10%   50%   90%
(Intercept)                           7.5    1.4   5.7   7.4   9.4
level                                -1.0    0.2  -1.3  -1.0  -0.8
tonetone51                          -13.6    2.3 -16.5 -13.5 -10.7
word_log_freq_z                       0.6    1.0  -0.6   0.6   1.8
level:tonetone51                      2.0    0.3   1.6   2.0   2.5
level:word_log_freq_z                -0.1    0.1  -0.2  -0.1   0.1
tonetone51:word_log_freq_z          -10.1    4.3 -15.6 -10.1  -4.6
level:tonetone51:word_log_freq_z      1.7    0.7   0.8   1.7   2.5
b[(Intercept) word:今晚]              0.1    0.6  -0.6   0.0   0.8
b[(Intercept) word:功課]  

In [12]:
sub_df <- df %>% 
      filter(lang == "man", participant == "P22")
model <- stan_glmer(acceptance ~ level * tone * word_log_freq_z + (1 | word), 
                          data = sub_df, 
                          family = binomial(link = "logit"),
                          adapt_delta = 0.99,
                          refresh = 0)
summary(model)


Model Info:
 function:     stan_glmer
 family:       binomial [logit]
 formula:      acceptance ~ level * tone * word_log_freq_z + (1 | word)
 algorithm:    sampling
 sample:       4000 (posterior sample size)
 priors:       see help('prior_summary')
 observations: 200
 groups:       word (10)

Estimates:
                                      mean   sd    10%   50%   90%
(Intercept)                           6.8    1.4   5.1   6.7   8.5
level                                -1.1    0.2  -1.3  -1.1  -0.8
tonetone51                          -13.1    2.2 -15.9 -13.1 -10.4
word_log_freq_z                       0.9    0.9  -0.3   0.9   2.1
level:tonetone51                      2.0    0.3   1.6   2.0   2.4
level:word_log_freq_z                -0.1    0.1  -0.3  -0.1   0.1
tonetone51:word_log_freq_z           -5.5    3.5 -10.0  -5.5  -0.9
level:tonetone51:word_log_freq_z      1.3    0.6   0.6   1.3   2.0
b[(Intercept) word:今晚]              0.6    0.8  -0.3   0.5   1.7
b[(Intercept) word:功課]  

In [13]:
sub_df <- df %>% 
      filter(lang == "man", participant == "P31")
model <- stan_glmer(acceptance ~ level * tone * word_log_freq_z + (1 | word), 
                          data = sub_df, 
                          family = binomial(link = "logit"),
                          adapt_delta = 0.99,
                          refresh = 0)
summary(model)


Model Info:
 function:     stan_glmer
 family:       binomial [logit]
 formula:      acceptance ~ level * tone * word_log_freq_z + (1 | word)
 algorithm:    sampling
 sample:       4000 (posterior sample size)
 priors:       see help('prior_summary')
 observations: 200
 groups:       word (10)

Estimates:
                                      mean   sd    10%   50%   90%
(Intercept)                           7.3    1.6   5.4   7.3   9.4
level                                -1.0    0.2  -1.3  -1.0  -0.8
tonetone51                          -11.6    2.2 -14.4 -11.7  -9.0
word_log_freq_z                      -1.3    1.1  -2.8  -1.3   0.0
level:tonetone51                      1.8    0.3   1.5   1.8   2.1
level:word_log_freq_z                 0.2    0.1   0.1   0.2   0.4
tonetone51:word_log_freq_z           -1.2    3.1  -5.3  -1.1   2.6
level:tonetone51:word_log_freq_z      0.6    0.5   0.0   0.6   1.3
b[(Intercept) word:今晚]              0.4    1.0  -0.9   0.3   1.6
b[(Intercept) word:功課]  

In [14]:
sub_df <- df %>% 
      filter(lang == "man", participant == "P39")
model <- stan_glmer(acceptance ~ level * tone * word_log_freq_z + (1 | word), 
                          data = sub_df, 
                          family = binomial(link = "logit"),
                          adapt_delta = 0.99,
                          refresh = 0)
summary(model)


Model Info:
 function:     stan_glmer
 family:       binomial [logit]
 formula:      acceptance ~ level * tone * word_log_freq_z + (1 | word)
 algorithm:    sampling
 sample:       4000 (posterior sample size)
 priors:       see help('prior_summary')
 observations: 200
 groups:       word (10)

Estimates:
                                      mean   sd    10%   50%   90%
(Intercept)                           7.2    1.5   5.4   7.1   9.1
level                                -1.1    0.2  -1.4  -1.1  -0.9
tonetone51                          -11.5    2.1 -14.2 -11.4  -8.8
word_log_freq_z                      -0.2    1.0  -1.4  -0.1   1.1
level:tonetone51                      2.2    0.3   1.8   2.1   2.6
level:word_log_freq_z                -0.1    0.1  -0.3  -0.1   0.1
tonetone51:word_log_freq_z           -7.9    3.9 -12.9  -7.7  -3.0
level:tonetone51:word_log_freq_z      1.1    0.6   0.4   1.1   1.9
b[(Intercept) word:今晚]              0.8    1.0  -0.4   0.7   2.1
b[(Intercept) word:功課]  